# Fine-tuning Médical — Phi-3.5-mini avec QLoRA 4-bit

> **Mission Expérimentale — TechCorp Hackathon**  
> Stack : HuggingFace Transformers · PEFT/LoRA · BitsAndBytes · TRL SFTTrainer  
> Dataset : [ruslanmv/ai-medical-chatbot](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot)  
> Durée estimée : ~35–50 min sur GPU T4

---
⚠️ **Disclaimer** : Modèle expérimental — ne pas utiliser pour des décisions médicales réelles.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU non détecté. Activez-le : Runtime > Change runtime type > T4 GPU')

gpu = torch.cuda.get_device_properties(0)
print(f'GPU         : {gpu.name}')
print(f'VRAM totale : {gpu.total_memory / 1e9:.1f} GB')
print(f'CUDA        : {torch.version.cuda}')

In [ ]:
# transformers>=4.47.0 requis — processing_class ajouté dans 4.47 (incompatible avec 4.45)
!pip install -q 'transformers>=4.47.0' 'bitsandbytes>=0.43.0' 'peft>=0.12.0' 'trl>=0.12.0' 'accelerate>=0.34.0' 'datasets>=2.20.0'
print('Installation terminée.')

In [ ]:
MODEL_NAME    = 'microsoft/Phi-3.5-mini-instruct'
DATASET_NAME  = 'ruslanmv/ai-medical-chatbot'
NUM_EXAMPLES  = 2000
MAX_SEQ_LEN   = 1024
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
BATCH_SIZE    = 2
GRAD_ACCUM    = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 1
OUTPUT_DIR    = './medical_lora_adapter'

SYSTEM_PROMPT = (
    'You are a knowledgeable medical assistant. '
    'Provide accurate, evidence-based medical information. '
    'Always recommend consulting a qualified healthcare professional '
    'for diagnosis and treatment decisions.'
)

print(f'Modèle  : {MODEL_NAME}')
print(f'Dataset : {DATASET_NAME} ({NUM_EXAMPLES} exemples)')
print(f'LoRA    : r={LORA_R}, alpha={LORA_ALPHA}')

## 1. Dataset

In [ ]:
from datasets import load_dataset

print(f'Chargement : {DATASET_NAME}...')
raw = load_dataset(DATASET_NAME, split='train')
print(f'Exemples totaux : {len(raw):,}')
print(f'Colonnes        : {raw.column_names}')

row = raw[0]
q = row.get('Patient', row.get('input', row.get('question', '')))
a = row.get('Doctor',  row.get('output', row.get('answer',  '')))
print(f'\nExemple Patient : {str(q)[:150]}')
print(f'Exemple Doctor  : {str(a)[:150]}')

In [ ]:
def format_example(example):
    q = example.get('Patient', example.get('input',  example.get('question', '')))
    a = example.get('Doctor',  example.get('output', example.get('answer',   '')))
    if not q or not a:
        return {'text': ''}
    text = (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{str(q).strip()}<|end|>\n'
        f'<|assistant|>\n{str(a).strip()}<|end|>'
    )
    return {'text': text}

subset  = raw.select(range(min(NUM_EXAMPLES, len(raw))))
dataset = subset.map(format_example, num_proc=2)
dataset = dataset.filter(lambda x: len(x['text']) > 100)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != 'text'])

split   = dataset.train_test_split(test_size=0.05, seed=42)
train_d = split['train']
val_d   = split['test']

print(f'Train : {len(train_d):,} exemples')
print(f'Val   : {len(val_d):,} exemples')
print(f"\nExemple formaté :\n{train_d[0]['text'][:350]}...")

## 2. Modèle + LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print('Chargement du tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print(f'Chargement de {MODEL_NAME} en 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='eager',
)
model = prepare_model_for_kbit_training(model)

total = sum(p.numel() for p in model.parameters())
print(f'Modèle chargé — {total/1e9:.2f}B paramètres')

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Paramètres entraînables : {trainable:,} ({100*trainable/total:.2f}%)')

## 3. Entraînement

In [ ]:
# TRL >= 0.14 : SFTConfig regroupe tout ; tokenizer -> processing_class ; max_seq_length -> max_length
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    # --- SFT-specific ---
    dataset_text_field          = 'text',
    max_length                  = MAX_SEQ_LEN,
    packing                     = False,
    # --- Training ---
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = 'cosine',
    logging_steps               = 25,
    save_steps                  = 200,
    save_total_limit            = 2,
    eval_strategy               = 'steps',
    eval_steps                  = 100,
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
    optim                       = 'paged_adamw_8bit',
    weight_decay                = 0.01,
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    report_to                   = 'none',
    seed                        = 42,
)

trainer = SFTTrainer(
    model           = model,
    processing_class = tokenizer,   # 'tokenizer' renommé 'processing_class' dans trl récent
    train_dataset   = train_d,
    eval_dataset    = val_d,
    args            = sft_config,
)

print('Démarrage de l\'entraînement...')
stats = trainer.train()
print(f'\nTerminé !')
print(f'  Loss  : {stats.training_loss:.4f}')
print(f'  Durée : {stats.metrics["train_runtime"]:.0f}s')

## 4. Test du modèle

In [ ]:
model.eval()

def generate(question, max_new_tokens=256):
    prompt = (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{question}<|end|>\n'
        f'<|assistant|>\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).replace('<|end|>', '').strip()

questions = [
    'What are the main symptoms of type 2 diabetes?',
    'How does hypertension affect the cardiovascular system?',
    'What is the difference between viral and bacterial pneumonia?',
    'Explain the mechanism of action of beta-blockers.',
    'What are the first-line treatments for major depressive disorder?',
]

print('=' * 65)
for q in questions:
    print(f'\nQ : {q}')
    print(f'R : {generate(q)}')
    print('-' * 65)

## 5. Sauvegarde et téléchargement

In [ ]:
import os, shutil
from google.colab import files

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

saved = [(f, os.path.getsize(os.path.join(OUTPUT_DIR, f))/1e6)
         for f in os.listdir(OUTPUT_DIR)]
print('Fichiers sauvegardés :')
for name, size in sorted(saved):
    print(f'  {name:<45} {size:.1f} MB')

shutil.make_archive('medical_lora_adapter', 'zip', OUTPUT_DIR)
zip_size = os.path.getsize('medical_lora_adapter.zip') / 1e6
print(f'\nArchive : medical_lora_adapter.zip ({zip_size:.0f} MB)')
files.download('medical_lora_adapter.zip')